# 고정 피벗 단진자 자유응답 시뮬레이션

Rotary arm을 움직이지 않고 진자를 아래쪽 평형점에서 $\theta_0$만큼 들어 올린 뒤 정지 상태로 놓는 실험을 모델링한다. 각도는 아래쪽 평형점이 $0$이고 radian으로 계산한다.

이 노트북은 다음 네 경우를 같은 초기조건으로 비교한다.

1. 무감쇠 선형 모델: 시간영역/Laplace 해석해와 수치해
2. 무감쇠 $\sin\theta$ 모델: Jacobi 타원함수 정확해와 수치해
3. 감쇠 선형 모델: 시간영역/Laplace 해석해와 수치해
4. 감쇠 $\sin\theta$ 모델: 수치해와 약감쇠 근사

전체 유도는 `docs/simple_pendulum_free_response.md`에 있다.

## 모델

$$J\ddot\theta+c\dot\theta+mgl_c\sin\theta=0$$

$$\omega_0^2=\frac{mgl_c}{J},\qquad 2\beta=\frac{c}{J}$$

균일 막대($J=mL^2/3$, $l_c=L/2$)이면 $\omega_0=\sqrt{3g/(2L)}$이다. `PENDULUM_KIND='point_mass'`로 바꾸면 $\omega_0=\sqrt{g/L}$인 이상적인 점질량 단진자가 된다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.integrate import solve_ivp
from scipy.special import ellipj, ellipk

plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True})

In [ ]:
# 물리/실험 파라미터: 실제 장치에 맞게 수정
g = 9.81                 # m/s^2
L = 0.235                # m, 피벗부터 막대 끝까지
PENDULUM_KIND = 'uniform_rod'  # 'uniform_rod' 또는 'point_mass'
theta0_deg = 25.0        # 손으로 들어 올리는 초기각
theta0 = np.deg2rad(theta0_deg)
beta = 0.20              # 1/s, c/(2J); 0이면 무감쇠
t_end = 12.0
t = np.linspace(0.0, t_end, 3001)

omega0 = np.sqrt(3*g/(2*L)) if PENDULUM_KIND == 'uniform_rod' else np.sqrt(g/L)
zeta = beta / omega0
print(f'omega0={omega0:.5f} rad/s, T_linear={2*np.pi/omega0:.5f} s, zeta={zeta:.5f}')

## 1. 감쇠 없음 — 선형화 모델

시간영역에서 $\ddot\theta+\omega_0^2\theta=0$의 특성근은 $\pm j\omega_0$이다. $\theta(0)=\theta_0$, $\dot\theta(0)=0$을 적용하면

$$\theta(t)=\theta_0\cos(\omega_0t).$$

Laplace 변환에서는

$$\Theta(s)=\theta_0\frac{s}{s^2+\omega_0^2}$$

이고 역변환 결과가 같다. 아래에서는 이 해석해와 `solve_ivp` 수치해를 비교한다.

In [ ]:
def integrate_pendulum(t_eval, theta_initial, omega0, beta=0.0, nonlinear=True):
    def rhs(_, state):
        theta, theta_dot = state
        restoring = np.sin(theta) if nonlinear else theta
        return theta_dot, -2*beta*theta_dot - omega0**2*restoring
    return solve_ivp(rhs, (t_eval[0], t_eval[-1]), [theta_initial, 0.0],
                     t_eval=t_eval, rtol=1e-10, atol=1e-12, method='DOP853')

theta_lin_exact = theta0 * np.cos(omega0*t)
sol_lin = integrate_pendulum(t, theta0, omega0, beta=0.0, nonlinear=False)
print('선형 무감쇠 최대 |해석-수치| =', np.max(np.abs(theta_lin_exact-sol_lin.y[0])))
plt.plot(t, np.rad2deg(theta_lin_exact), label='analytic (time/Laplace)')
plt.plot(t, np.rad2deg(sol_lin.y[0]), '--', label='numerical')
plt.xlabel('Time (s)'); plt.ylabel('Angle from down (deg)'); plt.legend(); plt.show()

## 2. 감쇠 없음 — $\sin\theta$ 비선형 모델

에너지 보존으로부터 $k=\sin(|\theta_0|/2)$라 두면, $|\theta_0|<\pi$인 진동의 정확한 해는

$$\theta(t)=2\operatorname{sgn}(\theta_0)\sin^{-1}\left[k\,\operatorname{cd}(\omega_0t,k)\right],$$

$$\operatorname{cd}(u,k)=\frac{\operatorname{cn}(u,k)}{\operatorname{dn}(u,k)},\qquad T=\frac{4K(k)}{\omega_0}.$$

SciPy의 `ellipj`는 modulus $k$ 자체가 아니라 parameter $m=k^2$를 받는 점에 주의한다.

In [ ]:
def nonlinear_undamped_exact(t_eval, theta_initial, omega0):
    if np.isclose(theta_initial, 0.0):
        return np.zeros_like(t_eval)
    if abs(theta_initial) >= np.pi:
        raise ValueError('This oscillatory closed form requires |theta0| < pi.')
    k = np.sin(abs(theta_initial)/2)
    sn, cn, dn, _ = ellipj(omega0*t_eval, k*k)
    return 2*np.sign(theta_initial)*np.arcsin(np.clip(k*cn/dn, -1.0, 1.0))

theta_nl_exact = nonlinear_undamped_exact(t, theta0, omega0)
sol_nl = integrate_pendulum(t, theta0, omega0, beta=0.0, nonlinear=True)
k = np.sin(abs(theta0)/2)
T_nl = 4*ellipk(k*k)/omega0
print(f'T_nonlinear={T_nl:.6f} s ({(T_nl/(2*np.pi/omega0)-1)*100:.3f}% longer)')
print('비선형 무감쇠 최대 |해석-수치| =', np.max(np.abs(theta_nl_exact-sol_nl.y[0])))
plt.plot(t, np.rad2deg(theta_lin_exact), label='linear analytic')
plt.plot(t, np.rad2deg(theta_nl_exact), label='nonlinear elliptic analytic')
plt.plot(t, np.rad2deg(sol_nl.y[0]), '--', label='nonlinear numerical')
plt.xlabel('Time (s)'); plt.ylabel('Angle from down (deg)'); plt.legend(); plt.show()

In [ ]:
# 무감쇠 비선형 수치해의 에너지 보존 확인 (질량/J로 정규화한 에너지)
energy = 0.5*sol_nl.y[1]**2 + omega0**2*(1-np.cos(sol_nl.y[0]))
print('relative energy drift =', np.ptp(energy)/energy[0])
plt.plot(t, (energy-energy[0])/energy[0])
plt.xlabel('Time (s)'); plt.ylabel('Relative energy error'); plt.show()

## 3. 점성 감쇠 — 선형화 모델

$$\ddot\theta+2\beta\dot\theta+\omega_0^2\theta=0.$$

부족감쇠($\beta<\omega_0$)이고 정지 상태에서 놓으면

$$\theta(t)=\theta_0e^{-\beta t}\left[\cos(\omega_dt)+\frac{\beta}{\omega_d}\sin(\omega_dt)\right],\quad \omega_d=\sqrt{\omega_0^2-\beta^2}.$$

Laplace 영역에서는 초기조건을 포함하여

$$\Theta(s)=\theta_0\frac{s+2\beta}{s^2+2\beta s+\omega_0^2}.$$

아래 함수는 부족/임계/과감쇠를 모두 처리한다.

In [ ]:
def linear_damped_exact(t_eval, theta_initial, omega0, beta):
    if beta < omega0 and not np.isclose(beta, omega0):
        wd = np.sqrt(omega0**2-beta**2)
        return theta_initial*np.exp(-beta*t_eval)*(np.cos(wd*t_eval)+(beta/wd)*np.sin(wd*t_eval))
    if np.isclose(beta, omega0):
        return theta_initial*(1+omega0*t_eval)*np.exp(-omega0*t_eval)
    root = np.sqrt(beta**2-omega0**2)
    r1, r2 = -beta+root, -beta-root
    return theta_initial*(-r2*np.exp(r1*t_eval)+r1*np.exp(r2*t_eval))/(r1-r2)

theta_dlin_exact = linear_damped_exact(t, theta0, omega0, beta)
sol_dlin = integrate_pendulum(t, theta0, omega0, beta=beta, nonlinear=False)
print('선형 감쇠 최대 |해석-수치| =', np.max(np.abs(theta_dlin_exact-sol_dlin.y[0])))
plt.plot(t, np.rad2deg(theta_dlin_exact), label='analytic (time/Laplace)')
plt.plot(t, np.rad2deg(sol_dlin.y[0]), '--', label='numerical')
plt.xlabel('Time (s)'); plt.ylabel('Angle from down (deg)'); plt.legend(); plt.show()

## 4. 점성 감쇠 — $\sin\theta$ 비선형 모델

$$\ddot\theta+2\beta\dot\theta+\omega_0^2\sin\theta=0.$$

이 일반 문제에는 무감쇠 경우와 같은 타원함수 정확 닫힌형 해가 없다. 대신 정확한 에너지 관계

$$\frac{d}{dt}\left[\frac12\dot\theta^2+\omega_0^2(1-\cos\theta)\right]=-2\beta\dot\theta^2$$

를 얻을 수 있고, 시간응답은 수치적으로 적분한다. 약한 감쇠에서는 진폭 포락선 $A(t)\approx A_0e^{-\beta t}$와 진폭 의존 주기 $4K(\sin(A/2))/\omega_0$를 해석 근사로 사용할 수 있다.

In [ ]:
sol_dnl = integrate_pendulum(t, theta0, omega0, beta=beta, nonlinear=True)
envelope = abs(theta0)*np.exp(-beta*t)
plt.plot(t, np.rad2deg(theta_dlin_exact), label='linear damped analytic')
plt.plot(t, np.rad2deg(sol_dnl.y[0]), label='nonlinear damped numerical')
plt.plot(t, np.rad2deg(envelope), 'k--', alpha=.6, label='small-angle envelope')
plt.plot(t, -np.rad2deg(envelope), 'k--', alpha=.6)
plt.xlabel('Time (s)'); plt.ylabel('Angle from down (deg)'); plt.legend(); plt.show()

energy_d = 0.5*sol_dnl.y[1]**2 + omega0**2*(1-np.cos(sol_dnl.y[0]))
print('Energy is non-increasing within tolerance:', np.all(np.diff(energy_d) <= 1e-10))

## 5. 초기각에 따른 선형화 오차

초기각을 바꾸며 정확 비선형 주기와 선형 주기의 차이를 계산한다. 큰 진폭에서는 $\sin\theta\approx\theta$ 오차 때문에 비선형 주기가 더 길어진다.

In [ ]:
amplitudes_deg = np.linspace(1, 170, 300)
modulus = np.sin(np.deg2rad(amplitudes_deg)/2)
period_ratio = 2*ellipk(modulus**2)/np.pi
plt.plot(amplitudes_deg, 100*(period_ratio-1))
plt.xlabel('Initial amplitude (deg)'); plt.ylabel('Period increase over linear (%)'); plt.show()

## 6. 선택 사항: 측정 CSV와 겹쳐 보기

`encoder_release_experiment.ipynb` 또는 `encoder_live_monitor.py`가 만든 CSV 경로를 지정한다. CSV 첫 줄은 `#` 메타데이터이므로 `comment='#'`로 읽는다. 측정의 첫 샘플이 실제 release 순간과 정확히 같지 않을 수 있으므로 필요하면 `TIME_SHIFT_S`를 조정한다. 모델 파라미터와 초기각도 측정에 맞게 바꿔야 한다.

In [ ]:
CSV_PATH = None  # 예: Path('data/pendulum_release_20260907_220000.csv')
TIME_SHIFT_S = 0.0

if CSV_PATH is not None and Path(CSV_PATH).exists():
    measured = pd.read_csv(CSV_PATH, comment='#')
    tm = measured['time_s'].to_numpy() - TIME_SHIFT_S
    mask = tm >= 0
    model = integrate_pendulum(tm[mask], np.deg2rad(measured.loc[mask, 'pendulum_deg'].iloc[0]),
                               omega0, beta=beta, nonlinear=True)
    plt.plot(tm, measured['pendulum_deg'], label='measured', alpha=.8)
    plt.plot(tm[mask], np.rad2deg(model.y[0]), label='nonlinear model')
    plt.xlabel('Time from release (s)'); plt.ylabel('Angle from down (deg)'); plt.legend(); plt.show()
else:
    print('CSV_PATH를 지정하면 측정 데이터와 모델을 비교합니다.')